In [1]:
from langchain_core.prompts import ChatPromptTemplate

# 1. 定义一个 Prompt (Runnable)
prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")

# Prompt 也可以调用 invoke/stream
print(prompt.invoke({"topic": "ice cream"}))


messages=[HumanMessage(content='Tell me a joke about ice cream', additional_kwargs={}, response_metadata={})]


In [3]:
from langchain_core.tools import tool

# 2. 定义一个简单的 Tool (Runnable)
@tool
def multiply(a: int, b: int) -> int:
    """Multiplies a and b."""
    return a * b

# Tool 也可以调用 invoke/batch
print(multiply.invoke({"a": 2, "b": 3}))

# Tool 也可以调用 batch (自动并行)
print(multiply.batch([{"a": 2, "b": 3}, {"a": 4, "b": 5}]))
# 输出: [6, 20]

6
[6, 20]


In [5]:
# 1 导入 os 与 dotenv
import os
from dotenv import load_dotenv

# 2 加载 .env 环境变量
load_dotenv(override=True)

# 3 读取密钥与地址
DeepSeek_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DeepSeek_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 4 可选：打印密钥
print(DeepSeek_API_KEY)  # 可以通过打印查看
print(DeepSeek_BASE_URL)  # 可以通过打印查看

sk-050a61a973294cb799e1f6999ca86f8e
https://api.deepseek.com


In [6]:
# 1 导入 OpenAI 客户端
from openai import OpenAI

# 2 初始化 DeepSeek API 客户端
client = OpenAI(api_key=DeepSeek_API_KEY, base_url=DeepSeek_BASE_URL)

# 3 创建对话消息并发起请求
response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是乐于助人的助手，请根据用户的问题给出回答"},
        {"role": "user", "content": "你好，请你介绍一下你自己。"},
    ],
)

# 4 打印模型最终的响应结果
print(response.choices[0].message.content)

你好！我是你的智能助手，很高兴认识你！我是一名由语言模型驱动的AI，主要功能是帮助解答问题、提供信息、分享知识，以及协助完成各种任务。我可以帮助你处理日常生活中的疑问，比如学习、工作、技术问题，甚至陪你聊聊天。如果有任何需要，尽管问我吧！😊


In [8]:
# 使用init_chat_model初始化DeepSeek模型
from langchain.chat_models import init_chat_model

# 1. 初始化模型（自动识别供应商）
model = init_chat_model(
    "deepseek-chat",                # 指定DeepSeek的聊天模型
    model_provider="deepseek",      # 指定模型提供商为deepseek
)

# 一行代码切换模型，业务代码0改动
# model = init_chat_model("gpt-4o", model_provider="openai")
# model = init_chat_model("claude-3-5-sonnet", model_provider="anthropic")

question = "你好，请你介绍一下你自己。"

result = model.invoke(question)
print(result.content)

你好呀！很高兴认识你！😊

我是 **DeepSeek**，由深度求索公司创造的AI助手。让我给你说说我的“本领”：

✨ **我是什么？**
- 一个纯文本模型，但别被“纯文本”骗了！我支持处理多种文件格式（图片、PDF、Word、Excel、PPT等），能从中读取文字信息帮你分析
- 我的知识截止到2025年5月，会努力保持信息准确

🎯 **我能做什么？**
- 回答各种问题，从日常闲聊到专业咨询
- 文件分析：上传文档，我能帮你提取要点、总结内容
- 链接阅读：给我网址，我能看看里面的内容
- 支持联网搜索（需要你手动开启哦）
- 超长上下文（1M！），可以一次处理整本书籍

🌟 **我的特色**
- 完全免费！没错，不收费
- 可以语音输入（App端）
- 热情细腻，会用心理解你的需求

📱 **怎么找到我？**
可以通过官方应用商店下载App，随时随地和我聊天！

有什么想问的、想聊的，或者需要帮助的，尽管开口！我会用我最好的状态陪伴你、帮助你。你今天想聊点什么呢？💬


In [10]:
# 1. 使用init_embeddings初始化嵌入模型
from langchain.embeddings import init_embeddings

# 2. 初始化OpenAI的text-embedding-3-small嵌入模型
embedding = init_embeddings(model="text-embedding-3-small",provider="openai")

# 3. 将文本转换为向量表示
res = embedding.embed_query("Hello world")

# 4. 打印向量的前10个元素
print(res[:10])

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [11]:
# 导入所需的消息类型
from langchain.messages import HumanMessage, SystemMessage

# 创建系统消息，设定模型角色为编程专家
system_msg = SystemMessage("你是一个编程专家。")

# 创建用户消息，请求生成一段3行的Python示例代码
human_msg = HumanMessage("给我写一段 3 行的 Python 示例。")

# 将系统消息和用户消息组合成消息列表
messages = [system_msg, human_msg]

# 调用模型，传入消息列表并获取响应
resp = model.invoke(messages)

# 提取并返回模型生成的内容
resp.content

'```python\n# 计算列表中所有偶数的平方和\nnumbers = [1, 2, 3, 4, 5, 6]\nresult = sum(x**2 for x in numbers if x % 2 == 0)\nprint(f"偶数的平方和为: {result}")\n```'

In [12]:
from langchain_core.prompts import PromptTemplate

# 创建一个带有{product}占位符变量的模板，{} 中的变量会被动态替换
prompt_template = PromptTemplate.from_template(
    "为生产{product}的公司起一个好名字？"
)

# 使用具体值格式化模板
formatted_prompt = prompt_template.format(product="智能水杯")
# 输出: "为生产智能水杯的公司起一个好名字？"

# 将格式化后的提示词直接传递给模型
# response = model.invoke(formatted_prompt)

print(f"打印生成的提示词：{formatted_prompt}")
print("=" * 60)
# print(response.content)

打印生成的提示词：为生产智能水杯的公司起一个好名字？
